### In questa esercitazione alleneremo un transformers encoder decoder su un dataset composto da dialoghi e corrispondenti riassunti

In [39]:
import pandas as pd
import re

In [40]:
def get_train_test_data(data_dir):
    # Get the train data
    train_data = pd.read_json(f"{data_dir}/train.json")
    train_data.drop(['id'], axis=1, inplace=True)

    # Get the test data
    test_data = pd.read_json(f"{data_dir}/test.json")
    test_data.drop(['id'], axis=1, inplace=True)

    return train_data, test_data

In [41]:
train_dataset, test_dataset= get_train_test_data('./data')

In [44]:
print(len(train_dataset))
train_dataset.head(5)
train_dataset.loc[0].summary

14732


'Amanda baked cookies and will bring Jerry some tomorrow.'

In [45]:
# Aggiungiamo i token di SOS e EOS e togliamo new lines
def preprocess(input_data):
    # Define the custom preprocessing function
    def preprocess_util(input_data):
        # Remove newlines and double spaces
        removed_newlines = re.sub("\n|\r|\t", " ",  input_data)
        removed_double_spaces = ' '.join(removed_newlines.split(' '))
        # Add start of sentence and end of sentence tokens
        s = '[SOS] ' + removed_double_spaces + ' [EOS]'
        return s

    # Apply the preprocessing to the train and test datasets
    input_data['summary'] = input_data.apply(lambda row : preprocess_util(row['summary']), axis = 1)
    input_data['dialogue'] = input_data.apply(lambda row : preprocess_util(row['dialogue']), axis = 1)

    document = input_data['dialogue']
    summary = input_data['summary']

    return list(document.values), list(summary.values)

In [46]:
document, summary = preprocess(train_dataset)
document_test, summary_test = preprocess(test_dataset)

In [49]:
summary[0]

'[SOS] Amanda baked cookies and will bring Jerry some tomorrow. [EOS]'

In [56]:
#Creiamo il tokenizer
import json
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing
# Inizializza un tokenizer BPE

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

# Configura il trainer
trainer = BpeTrainer(
    vocab_size=12000, # Per un dataset piccolo come il nostro 10 mila è sufficiente
    min_frequency=2,
    special_tokens=["[PAD]", "[SOS]", "[EOS]", "[UNK]"]
)

texts = document + summary + document_test + summary_test
# Avvia l'addestramento sui tuoi dati
tokenizer.train_from_iterator(texts, trainer)
# Salva il tokenizer per non doverlo rifare
tokenizer.save("tokenizer_summarization.json")

In [51]:
### Proviamo il tokenizer che abbiamo allenato
# Testiamo il tokenizer su una riga del dataset
test_text = document[0]#"[SOS] Olivia: Who are you voting for in this election?   Oliver: Liberals as always.  Olivia: Me too!!  Oliver: Great [EOS]"
encoded = tokenizer.encode(test_text)

print(f"Testo originale: {test_text}")
print(f"Tokens: {encoded.tokens}")
print(f"IDs: {encoded.ids}")

# Decoding per verifica
print(f"Reconstructed: {tokenizer.decode(encoded.ids)}")

Testo originale: [SOS] Amanda: I baked  cookies. Do you want some?  Jerry: Sure!  Amanda: I'll bring you tomorrow :-) [EOS]
Tokens: ['[SOS]', 'Amanda', ':', 'I', 'baked', 'cookies', '.', 'Do', 'you', 'want', 'some', '?', 'Jerry', ':', 'Sure', '!', 'Amanda', ':', 'I', "'", 'll', 'bring', 'you', 'tomorrow', ':-)', '[EOS]']
IDs: [1, 1639, 29, 44, 7150, 3963, 17, 1029, 627, 760, 702, 34, 2063, 29, 1148, 4, 1639, 29, 44, 10, 620, 1179, 627, 876, 1822, 2]
Reconstructed: Amanda : I baked cookies . Do you want some ? Jerry : Sure ! Amanda : I ' ll bring you tomorrow :-)


In [57]:
test_phrase = "Robert: I need the wrench."
encoded = tokenizer.encode(test_phrase)
print(encoded.tokens)

['Robert', ':', 'I', 'need', 'the', 'w', 'rench', '.']


In [58]:
import torch
from torch.utils.data import Dataset, DataLoader

class SummarizationDataset(Dataset):
    def __init__(self, dialogue,summary, tokenizer, max_len_input=512, max_len_target=128):
        self.dialogue = dialogue
        self.summary = summary
        self.tokenizer = tokenizer
        self.max_len_input = max_len_input
        self.max_len_target = max_len_target

    def __len__(self):
        return len(self.dialogue)

    def __getitem__(self, idx):


        # Tokenizzazione
        tokenized_input = self.tokenizer.encode(self.dialogue[idx])
        tokenized_target = self.tokenizer.encode(self.summary[idx])

        # Padding manuale per l'input (Encoder)
        input_ids = self._pad_sequence(tokenized_input.ids, self.max_len_input)

        # Padding manuale per il target (Decoder)
        target_ids = self._pad_sequence(tokenized_target.ids, self.max_len_target)

        return {
            "encoder_input": torch.tensor(input_ids, dtype=torch.long),
            "decoder_input": torch.tensor(target_ids, dtype=torch.long),
            # Creiamo anche i target per la loss (spostati di 1)
            "labels": torch.tensor(target_ids, dtype=torch.long)
        }

    def _pad_sequence(self, tokens, max_len):
        # Taglia se troppo lungo, aggiunge [PAD] (ID 0) se troppo corto
        if len(tokens) > max_len:
            return tokens[:max_len]
        return tokens + [0] * (max_len - len(tokens))

# Inizializzazione
train_dataset_py = SummarizationDataset(dialogue=document, summary=summary, tokenizer=tokenizer, max_len_input=512, max_len_target=128)
test_dataset_py = SummarizationDataset(dialogue=document_test, summary=summary_test, tokenizer=tokenizer, max_len_input=512, max_len_target=128)

In [59]:
#Definisco i dataloader
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset_py,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset_py,
    batch_size=BATCH_SIZE,
    shuffle=True
)
# Testiamo un batch
example_batch = next(iter(train_loader))
print(f"Shape Input Encoder: {example_batch['encoder_input'].shape}") # [8, 512]
print(f"Shape Input Decoder: {example_batch['decoder_input'].shape}") # [8, 128]

Shape Input Encoder: torch.Size([16, 512])
Shape Input Decoder: torch.Size([16, 128])


In [60]:
#Definisco un layer di encoding
import torch.nn as nn

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        # Multi-Head Attention definita con le classi di PyTorch
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
                                               # Il dropout del multihead attention si usa sulle attention matrices dopo aver fatto la softmax

        # Feed Forward Network
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

        # Normalizzazione
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # 1. Self-Attention + Residual Connection
        #La mask serve a "oscurare" i token di padding
        attn_output, _ = self.self_attn(x, x, x, key_padding_mask=mask)
        x = self.norm1(x + self.dropout(attn_output)) #Qui il dropout è sull output intero del multihead attention

        # 2. Feed Forward + Residual Connection
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))

        return x

## Positional encoding formula:
$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{model}})$$$$PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{model}})$$

In [14]:
### Definiamo il positional encoding
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        # Creiamo una matrice di zeri (max_len, d_model)
        pe = torch.zeros(max_len, d_model)

        # Vettore delle posizioni (0, 1, 2, ..., max_len)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # Calcoliamo il termine di divisione (il denominatore della formula)
        # Usiamo il logspace per stabilità numerica
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # Applichiamo il seno alle posizioni pari e il coseno a quelle dispari
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Aggiungiamo una dimensione per il batch (1, max_len, d_model)
        pe = pe.unsqueeze(0)

        # register_buffer indica a PyTorch che questo non è un parametro da allenare
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: [batch_size, seq_len, d_model]
        # Sommiamo il positional encoding all'embedding del testo
        x = x + self.pe[:, :x.size(1), :]
        return x

In [15]:
# Definiamo l'intero encoder
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_len, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)

        # Usiamo il Positional Encoding sinusoidale
        self.pos_encoding = PositionalEncoding(d_model, max_len)

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        # 1. Embedding
        x = self.embedding(x) * math.sqrt(x.size(-1)) # Scaling come nel paper moltiplico gli embedding per la radice quadrata di d_k in modo tale che i pos encoding non influiscano più di tanto essendo compresi tra -1 e 1

        # 2. Aggiunta Positional Encoding
        x = self.pos_encoding(x)

        x = self.dropout(x)

        # 3. Passaggio attraverso i blocchi
        for layer in self.layers:
            x = layer(x, mask)

        return x

In [16]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        # 1. Self-Attention (per il riassunto generato finora)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)

        # 2. Cross-Attention (per guardare l'output dell'Encoder)
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask, tgt_mask):
        # x: input del decoder (riassunto)
        # enc_output: output dell'encoder (dialogo)

        # A. Masked Self-Attention
        # attn_mask è la maschera causale (look-ahead mask)
        # key_padding_mask evita di guardare i [PAD] nel target
        attn1, _ = self.self_attn(x, x, x, attn_mask=tgt_mask) #tgt_mask è la maschera che abbiamo visto nelle slides che oscura la parte triangolare superiore della matrice di attenzione
        x = self.norm1(x + self.dropout(attn1))

        # B. Cross-Attention
        # Query = x (dal decoder)
        # Key e Value = enc_output (dall'encoder)
        attn2, _ = self.cross_attn(x, enc_output, enc_output, key_padding_mask=src_mask)
        x = self.norm2(x + self.dropout(attn2))

        # C. Feed Forward
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))

        return x

In [61]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_len, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)

        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) # di solito d_model * n_heads = 512
            for _ in range(n_layers)
        ])

        self.fc_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask, tgt_mask):
        x = self.embedding(x) * math.sqrt(x.size(-1))
        x = self.pos_encoding(x)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)

        return self.fc_out(x)

In [62]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_len_src, max_len_tgt, dropout):
        super().__init__()

        self.encoder = Encoder(vocab_size, d_model, n_layers, n_heads, d_ff, max_len_src, dropout)
        self.decoder = Decoder(vocab_size, d_model, n_layers, n_heads, d_ff, max_len_tgt, dropout)

    def make_src_mask(self, src):
        # Maschera per ignorare il [PAD] (ID = 0) nell'input del dialogo
        # src shape: (batch_size, src_len)
        src_mask = (src == 0) # Ritorna True dove c'è il padding, 0 è l'indice del token [PAD}
        return src_mask

    def make_tgt_mask(self, tgt):
        # 1. Padding mask per il target
        tgt_pad_mask = (tgt == 0)

        # 2. Causal mask (Look-ahead)
        tgt_len = tgt.size(1)
        # Crea una matrice triangolare superiore di True (per nascondere il futuro)
        tgt_mask = torch.triu(torch.ones((tgt_len, tgt_len), device=tgt.device), diagonal=1).bool()
        return tgt_mask

    def forward(self, src, tgt):
        # Creazione maschere
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)

        # Passaggio nell'Encoder
        enc_output = self.encoder(src, src_mask)

        # Passaggio nel Decoder
        # Nota: il decoder riceve l'output dell'encoder e le due maschere
        output = self.decoder(tgt, enc_output, src_mask, tgt_mask)

        return output

In [63]:
#Inizializziamo variabili e modello:
VOCAB_SIZE = 12000 # Quello scelto per il tokenizer
D_MODEL = 256     # Dimensione degli embedding
N_LAYERS = 4    # Numero di blocchi encoder/decoder
N_HEADS = 8     # Teste di attenzione
D_FF = 512        # Dimensione feed-forward interna
MAX_LEN_SRC = 512 # Lunghezza max dialogo
MAX_LEN_TGT = 128 # Lunghezza max riassunto
DROPOUT = 0.3

model = Transformer(VOCAB_SIZE, D_MODEL, N_LAYERS, N_HEADS, D_FF, MAX_LEN_SRC, MAX_LEN_TGT, DROPOUT)

# Spostiamolo sulla GPU se disponibile
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Transformer(
  (encoder): Encoder(
    (embedding): Embedding(12000, 256)
    (pos_encoding): PositionalEncoding()
    (layers): ModuleList(
      (0-3): 4 x EncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (feed_forward): Sequential(
          (0): Linear(in_features=256, out_features=512, bias=True)
          (1): ReLU()
          (2): Dropout(p=0.3, inplace=False)
          (3): Linear(in_features=512, out_features=256, bias=True)
        )
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.3, inplace=False)
      )
    )
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(12000, 256)
    (pos_encoding): PositionalEncoding()
    (layers): ModuleList(
      (0-3): 4 x DecoderLayer(
        (self_a

In [64]:
def evaluate_per(model, dataloader, criterion, device):
    model.eval() # Imposta il modello in modalità valutazione (disabilita Dropout)
    total_loss = 0

    with torch.no_grad(): # Disabilita il calcolo dei gradienti per risparmiare memoria
        for batch in dataloader:
            src = batch['encoder_input'].to(device)
            tgt = batch['decoder_input'].to(device)

            tgt_input = tgt[:, :-1]
            tgt_expected = tgt[:, 1:]

            output = model(src, tgt_input)

            loss = criterion(output.view(-1, VOCAB_SIZE), tgt_expected.reshape(-1))
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [65]:
import torch.optim as optim
import time
from torch.optim.lr_scheduler import LinearLR, ExponentialLR, SequentialLR

criterion = nn.CrossEntropyLoss(ignore_index=0) # Ignora il Padding
optimizer = optim.AdamW(lr=1e-4, params = model.parameters(), weight_decay=5e-3)
EPOCHS = 20

# 2. Scheduler 1: Warmup lineare (va da 0.1 * lr a lr in 5 epoche)
# 1. Calcolo step
steps_per_epoch = len(train_loader)

total_steps = EPOCHS * steps_per_epoch
steps_per_epoch = len(train_loader)
warmup_steps = int(total_steps * 0.1) 

# 2. Scheduler 1: Warmup (Sale per i primi warmup_steps)
scheduler1 = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_steps)

# 3. Scheduler 2: Decadimento Lineare (Invece di esponenziale per evitare il crollo precoce)
# Scende dal valore massimo a quasi zero per il resto dei passi
scheduler2 = LinearLR(optimizer, start_factor=1.0, end_factor=0.1, total_iters=total_steps - warmup_steps)

scheduler = SequentialLR(
    optimizer, 
    schedulers=[scheduler1, scheduler2], 
    milestones=[warmup_steps]
)
train_losses = []
val_losses = []

best_val_loss = float('inf')

print(f"Inizio training su device: {device}")

for epoch in range(EPOCHS):
    start_time = time.time()

    # --- TRAINING ---
    model.train()
    total_train_loss = 0

    for batch in train_loader:
        # Sposta i dati su GPU
        src = batch['encoder_input'].to(device)
        tgt = batch['decoder_input'].to(device)
        # Teacher Forcing: input fino alla penultima, target dalla seconda
        tgt_input = tgt[:, :-1]
        tgt_expected = tgt[:, 1:]

        optimizer.zero_grad()

        # Maschere gestite internamente alla classe Transformer (forward)
        output = model(src, tgt_input)
        # Reshape per la Loss: (Batch * Seq_Len, Vocab_Size) vs (Batch * Seq_Len)
        loss = criterion(output.reshape(-1, VOCAB_SIZE), tgt_expected.reshape(-1))

        loss.backward()

        # Clipping dei gradienti (Vitale per i Transformer!)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step() 
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- VALIDATION ---
    # Usiamo la funzione evaluate che abbiamo scritto prima
    avg_val_loss = evaluate_per(model, test_loader, criterion, device)
    val_losses.append(avg_val_loss)

    # --- SCHEDULER STEP ---
    # Lo scheduler guarda la loss di validazione e decide se abbassare il LR
    current_lr = optimizer.param_groups[0]['lr']

    end_time = time.time()
    epoch_mins, epoch_secs = divmod(end_time - start_time, 60)

    # --- CHECKPOINTING ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model_transformer.pt')
        save_msg = "-> Modello Salvato!"
    else:
        save_msg = ""
    dialogue_text = train_dataset.dialogue.iloc[0]#'Robert: I need the wrench. Sarah: It is in the docking bay. Robert: Found it, thanks.'## #
    summary_true = summarize(model, dialogue_text, tokenizer, device)
    print(dialogue_text)
    print(f"Generato: {summary_true}")
    print(f'Epoch: {epoch+1:02} | Time: {int(epoch_mins)}m {int(epoch_secs)}s | LR: {current_lr:.8f}')
    print(f'\tTrain Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} {save_msg}')

Inizio training su device: cuda
[SOS] Amanda: I baked  cookies. Do you want some?  Jerry: Sure!  Amanda: I'll bring you tomorrow :-) [EOS]
Generato: lyna the smoking him in at raised . contribut activities Get it yesterday .
Epoch: 01 | Time: 1m 23s | LR: 0.00005500
	Train Loss: 7.0935 | Val Loss: 6.1372 -> Modello Salvato!
[SOS] Amanda: I baked  cookies. Do you want some?  Jerry: Sure!  Amanda: I'll bring you tomorrow :-) [EOS]
Generato: roat will India for solution when .
Epoch: 02 | Time: 1m 37s | LR: 0.00010000
	Train Loss: 5.9036 | Val Loss: 5.6014 -> Modello Salvato!
[SOS] Amanda: I baked  cookies. Do you want some?  Jerry: Sure!  Amanda: I'll bring you tomorrow :-) [EOS]
Generato: loud will bring child .
Epoch: 03 | Time: 1m 37s | LR: 0.00009500
	Train Loss: 5.5009 | Val Loss: 5.3551 -> Modello Salvato!


KeyboardInterrupt: 

In [27]:
import torch.nn.functional as F
def summarize(model, dialogue, tokenizer, device, max_len=50, temperature=1.0):
    model.eval()
    # 1. Prepara l'input dell'encoder
    src_tokens = tokenizer.encode("[SOS] " + dialogue + " [EOS]").ids
    src_tensor = torch.tensor([src_tokens]).to(device)

    # 2. L'encoder lavora una volta sola
    with torch.no_grad():
        enc_output = model.encoder(src_tensor, model.make_src_mask(src_tensor))

    # 3. Il Decoder inizia con il token di START
    # Partiamo con una lista che contiene solo l'ID di [SOS]
    outputs = [tokenizer.token_to_id("[SOS]")]

    for i in range(max_len):
        tgt_tensor = torch.tensor([outputs]).to(device)

        # 4. Predizione (il decoder guarda l'enc_output e quello scritto finora)
        with torch.no_grad():
            output = model.decoder(tgt_tensor, enc_output,
                                   model.make_src_mask(src_tensor),
                                   model.make_tgt_mask(tgt_tensor))

        # Prendiamo l'ultimo token predetto (l'ultima posizione della sequenza)
        logits = output[:, -1, :] / temperature
        
        # Trasformiamo in probabilità
        probs = F.softmax(logits, dim=-1)
        
        # Campioniamo dalla distribuzione invece di prendere il massimo secco
        next_word = torch.multinomial(probs, num_samples=1).item()

        outputs.append(next_word)
        # 5. Se il modello predice [EOS], abbiamo finito
        if next_word == tokenizer.token_to_id("[EOS]"):
            break

    # Trasforma gli ID in parole leggibili
    return tokenizer.decode(outputs)

In [ ]:
from tqdm import tqdm
import evaluate

# Carichiamo la metrica
rouge_metric = evaluate.load("rouge")
def calculate_rouge(model, dataset, tokenizer, device, num_samples=10):
    model.eval()

    predictions = []
    references = []

    # Prendiamo un sottoinsieme random dal dataset (o i primi N)
    # Assumiamo che dataset sia il DataFrame o una lista di dict
    samples =  dataset[:num_samples]

    print(f"Calcolo ROUGE su {num_samples} esempi...")

    for _, row in tqdm(samples.iterrows(), total=num_samples):
        dialogue = row['dialogue']
        summary_true = row['summary']

        # Generiamo il riassunto con la nostra funzione summarize() creata prima
        # Nota: summarize() restituisce una stringa decodificata

        summary_pred = summarize(model, dialogue, tokenizer, device)

        # Pulizia post-generazione (rimuoviamo token speciali se rimasti)
        summary_pred = summary_pred.replace("[SOS]", "").replace("[EOS]", "").strip()

        predictions.append(summary_pred)
        references.append(summary_true)

    # Calcolo effettivo
    results = rouge_metric.compute(predictions=predictions, references=references)

    return results

results = calculate_rouge(model, test_dataset, tokenizer, device, num_samples=100)


In [69]:
sum(p.numel() for p in model.parameters())

14499552

In [66]:
#Carichiamo il modello migliore
# 1. Ridefinisci l'architettura (deve essere IDENTICA a quella del training)
# Assicurati di usare gli stessi iperparametri (d_model, n_layers, ecc.)
model = Transformer(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=256,
    n_layers=4,
    n_heads=8,
    d_ff=512,
    max_len_src=512,
    max_len_tgt=128,
    dropout=0.4
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

checkpoint_path = 'best_of_the_best_model_transformer.pt'

if torch.cuda.is_available():
   model.load_state_dict(torch.load(checkpoint_path))
else:
   # Se carichi su CPU un modello allenato su GPU
   model.load_state_dict(torch.load(checkpoint_path, map_location=torch.device('cpu')))

print("Modello caricato con successo!")

dialogue_text = train_dataset.dialogue.iloc[0]#'Robert: I need the wrench. Sarah: It is in the docking bay. Robert: Found it, thanks.'## #
summary_true = summarize(model, dialogue_text, tokenizer, device)
print(dialogue_text)
print()
print(f"Generato: {summary_true}")


Modello caricato con successo!
[SOS] Amanda: I baked  cookies. Do you want some?  Jerry: Sure!  Amanda: I'll bring you tomorrow :-) [EOS]

Generato: Amanda will bring Jerry a lift tomorrow .


In [ ]:
print(len(train_loader))